In [1]:
import sys
sys.path.append("/kaggle/input/datasets/aithbb/corruptdecoder/custom_archive (37)/starter_kit")
from data_loader import load_data
from load_utility import load_base_inception, load_inception_transforms
from model import load_decoder
from submission import submit_task1, submit_task2
embs1, imgs1 = load_data('/kaggle/input/datasets/aithbb/corruptdecoder/train_data (8)/problem_data/', 'subtask1', 'ordered')
embs2, imgs2 = load_data('/kaggle/input/datasets/aithbb/corruptdecoder/train_data (8)/problem_data/', 'subtask2', 'train')
print(embs1.shape)
print(imgs1.shape)
print(imgs1.min(), imgs1.max())
print(embs2.shape)
print(imgs2.shape)
print(imgs2.min(), imgs2.max())

torch.Size([500, 128])
torch.Size([500, 3, 160, 160])
tensor(0.) tensor(1.)
torch.Size([4000, 128])
torch.Size([4000, 3, 160, 160])
tensor(0.) tensor(1.)


In [2]:
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import torch.nn as nn
import torch
from tqdm.auto import tqdm
decoder = load_decoder('/kaggle/input/datasets/aithbb/corruptdecoder/custom_archive (37)/starter_kit/decoder.pt', 'cuda')
decoder.eval()
if 'embs11' in locals():
    trainset = TensorDataset(torch.cat([embs2[:400], embs1]), torch.cat([imgs2[:400], imgs1]))
    print("using new data")
else:
    trainset = TensorDataset(torch.cat([embs2[:400], embs1]), torch.cat([imgs2[:400], imgs1]))
valset = TensorDataset(embs2[400:], imgs2[400:])
trainloader = DataLoader(trainset, batch_size=32)
valloader = DataLoader(valset, batch_size=32)
optimizer = optim.Adam(decoder.parameters(), lr = 0.00001)
criterion = nn.L1Loss()
class InceptionMid(nn.Module):
    def __init__(self, weights_path, num_blocks: int) -> None:
        super().__init__()
        base = load_base_inception(weights_path)
        self.backbone = nn.Sequential(*list(base.children())[:num_blocks]).eval()
        self.transform = load_inception_transforms()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(self.transform(x))
incept = InceptionMid('/kaggle/input/datasets/aithbb/corruptdecoder/custom_archive (37)/starter_kit/inception_v3.pth', num_blocks=13).to('cuda').eval()
for e in range(5):
    decoder.train()
    rl = 0
    for data in tqdm(trainloader):
        data[0] = data[0].to('cuda')
        data[1] = data[1].to('cuda')
        optimizer.zero_grad()
        outputs = (decoder(data[0]))
        data[1] = (data[1])
        loss = criterion(outputs.flatten(1), data[1].flatten(1))
        # print(outputs.flatten(1).shape)
        rl += loss.item()
        loss.backward()
        optimizer.step()
    print(e + 1, rl / len(trainloader), end = ' ')
    decoder.eval()
    rl = 0
    for data in valloader:
        data[0] = data[0].to('cuda')
        data[1] = data[1].to('cuda')
        outputs = (decoder(data[0]))
        data[1] = (data[1])
        loss = criterion(outputs.flatten(1), data[1].flatten(1))
        # print(outputs.flatten(1).shape)
        rl += loss.item()
    print(rl / len(trainloader))
embstest2 = load_data('/kaggle/input/datasets/aithbb/corruptdecoder/train_data (8)/problem_data/', 'subtask2', 'test')[0]

  0%|          | 0/29 [00:00<?, ?it/s]

1 0.18945895443702565 0.4675473989597682


  0%|          | 0/29 [00:00<?, ?it/s]

2 0.1648998936188632 0.40727789288964766


  0%|          | 0/29 [00:00<?, ?it/s]

3 0.1572030746731265 0.3908232458192727


  0%|          | 0/29 [00:00<?, ?it/s]

4 0.1536334049085091 0.3788052661151722


  0%|          | 0/29 [00:00<?, ?it/s]

5 0.15074604184463106 0.3720083719697492


In [3]:
import numpy as np
decoder.eval()
res = []
embstest2.shape
for l in tqdm(range(0, 500, 16)):
    res.append(incept(decoder(embstest2[l:l+16].to('cuda'))).cpu().detach().numpy())
    # print(res[-1].shape)
    # break
res = np.concatenate(res)
print(res.shape)

  0%|          | 0/32 [00:00<?, ?it/s]

(500, 768, 17, 17)


In [4]:
embs11, imgs11 = load_data('/kaggle/input/datasets/aithbb/corruptdecoder/train_data (8)/problem_data/', 'subtask1', 'unordered')
ress = []
for l in tqdm(range(0, len(embs11), 16)):
    ress.append(incept(decoder(embs11[l:l+16].to('cuda'))).cpu().detach().numpy())
    # print(res[-1].shape)
    # break
ress = np.concatenate(ress)
res2 = []
for l in tqdm(range(0, len(embs11), 16)):
    res2.append(incept(imgs11[l:l+16].to('cuda')).cpu().detach().numpy())
    # print(res2[-1].shape)
    # break
res2 = np.concatenate(res2)
print(ress.shape, res2.shape)


  0%|          | 0/94 [00:00<?, ?it/s]

  0%|          | 0/94 [00:00<?, ?it/s]

(1500, 768, 17, 17) (1500, 768, 17, 17)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
if 'ans' not in locals():
    ans = linear_sum_assignment(-cosine_similarity(ress.reshape((ress.shape[0], -1)), res2.reshape((res2.shape[0], -1))))[1]

In [6]:
import pandas as pd
pd.concat([submit_task1(ans), submit_task2(res)]).to_csv('submission.csv', index=False)
